In [ ]:
!pip install qiskit==0.43.1 qiskit-ibm-provider==0.5.0

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 📌 Load Data
df = pd.read_csv("/content/drive/MyDrive/QML/validation/processed_gait_features.csv")

df.head()

,HipAngles_t0,KneeAngles_t0,AnkleAngles_t0,HipAngles_t1,KneeAngles_t1,AnkleAngles_t1,HipAngles_t2,KneeAngles_t2,AnkleAngles_t2,HipAngles_t3,...,HipAngles_t9,KneeAngles_t9,AnkleAngles_t9,HipAngles_t10,KneeAngles_t10,AnkleAngles_t10,HipAngles_t11,KneeAngles_t11,AnkleAngles_t11,Label
0,30.049283,11.570827,-2.482850,30.037282,11.714903,-2.519216,30.024681,11.858510,-2.555104,30.011461,...,29.917874,12.850300,-2.792050,29.899789,12.990142,-2.823833,29.881429,13.129831,-2.855325,0
1,29.967856,12.428170,-2.693670,29.951934,12.569373,-2.727010,29.935280,12.710084,-2.759807,29.917874,...,29.803088,13.684542,-2.976271,29.781543,13.821460,-3.004636,29.758920,13.957372,-3.032007,0
2,29.862726,13.269265,-2.886438,29.843533,13.408295,-2.917049,29.823703,13.546771,-2.947034,29.803088,...,29.655016,14.491002,-3.131655,29.625860,14.622537,-3.154451,29.595627,14.753186,-3.176210,0
3,29.735071,14.092141,-3.058261,29.709800,14.225834,-3.083470,29.683023,14.358725,-3.107948,29.655016,...,29.465393,15.264074,-3.250358,29.430696,15.388425,-3.265330,29.394712,15.511519,-3.279152,0
4,29.564392,14.882804,-3.196806,29.532227,15.011250,-3.216112,29.499206,15.138379,-3.234003,29.465393,...,29.231656,15.993528,-3.323775,29.186913,16.110906,-3.331378,29.140990,16.226755,-3.337219,0


# Step 1: Quantum Feature Extraction

In [ ]:
from google.colab import userdata
QIBM_API_KEY = userdata.get('QIBM_API_KEY')

In [ ]:
from qiskit_ibm_provider import IBMProvider

IBMProvider.save_account(QIBM_API_KEY, overwrite=True)
provider = IBMProvider()

In [ ]:
from qiskit_ibm_provider import least_busy

backend = least_busy(
    provider.backends(
        simulator=False,
        min_num_qubits=4,
        operational=True,
        dynamic_reprate_enabled=True
    )
)

print(f"Running on backend: {backend.name}")

Running on backend: ibm_brisbane


In [ ]:
from qiskit.primitives import Sampler
from qiskit import QuantumCircuit

# Define quantum circuit
num_qubits = 5
features = np.random.uniform(0, np.pi, num_qubits)

qc = QuantumCircuit(num_qubits, num_qubits)

for i in range(num_qubits):
    qc.ry(features[i], i)

for i in range(num_qubits - 1):
    qc.cx(i, i + 1)
qc.cx(num_qubits - 1, 0)

# ✅ Explicitly map qubit i → classical bit i
qc.measure(range(num_qubits), range(num_qubits))

sampler = Sampler()
job = sampler.run([qc])
result = job.result()

print("Quantum output:", result.quasi_dists[0])

Quantum output: {0: 5.2721406035e-05, 1: 4.09042347743e-05, 2: 2.37984375e-08, 3: 9.33445205e-08, 4: 6.7662778e-09, 5: 5.2496593e-09, 6: 2.9650312209e-06, 7: 1.1629730607e-05, 8: 0.0282758557475162, 9: 0.0219380006894219, 10: 1.27637185246e-05, 11: 5.00630843063e-05, 12: 0.0002269524793196, 13: 0.0001760825098359, 14: 0.0994521955002785, 15: 0.39008096568147, 16: 0.1734903052904498, 17: 0.2236113908206172, 18: 0.0003959093585167, 19: 0.0001009381599915, 20: 2.22657872191e-05, 21: 2.86983393075e-05, 22: 0.0493260789398215, 23: 0.012575817016386, 24: 5.1723762269e-06, 25: 6.6666678579e-06, 26: 1.18034962e-08, 27: 3.0093332e-09, 28: 4.15154052e-08, 29: 5.35091427e-08, 30: 9.19703459771e-05, 31: 2.34480880459e-05}


In [ ]:
def quantum_feature_vector(features, backend_name="ibm_brisbane", num_qubits=5):
    from qiskit_ibm_provider import IBMProvider
    from qiskit.primitives import Sampler
    from qiskit import QuantumCircuit
    import numpy as np

    provider = IBMProvider()
    qc = QuantumCircuit(num_qubits, num_qubits)

    for i in range(num_qubits):
        qc.ry(features[i], i)

    for i in range(num_qubits - 1):
        qc.cx(i, i + 1)
    qc.cx(num_qubits - 1, 0)

    qc.measure(range(num_qubits), range(num_qubits))

    sampler = Sampler()
    job = sampler.run([qc])
    result = job.result()

    vec = [float(result.quasi_dists[0].get(i, 0.0)) for i in range(2 ** num_qubits)]
    return vec

In [ ]:
import json

# --- CONFIG ---
CACHE_FILE = "/content/drive/MyDrive/QML/validation/stroke_quantum_cache.json"
num_qubits = 5

# --- Hashing utility ---
def hash_input(arr):
    return hashlib.md5(np.round(arr, 6).tobytes()).hexdigest()

# --- Load existing cache ---
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE, "r") as f:
        cache = json.load(f)
else:
    cache = {}

# --- Safe quantum feature wrapper ---
def cached_quantum_feature_vector(features):
    key = hash_input(features)
    if key in cache:
        return cache[key]

    try:
        vec = quantum_feature_vector(features, num_qubits=num_qubits)
        cache[key] = vec

        # Save after each success
        with open(CACHE_FILE, "w") as f:
            json.dump(cache, f)

        print(f"✅ Processed: {key}")
        return vec

    except Exception as e:
        print(f"❌ Error on input {key}: {e}")
        return [0.0] * (2 ** num_qubits)  # placeholder if fail

# Step 2: Apply Quantum Encoding to Entire Dataset

In [ ]:
def extract_quantum_features_deduplicated(
    df,
    gait_features,
    quantum_feature_fn,
    num_qubits=5,
    cache_file="/content/drive/MyDrive/QML/validation/stroke_quantum_cache.json",
    save_every=10,
    output_csv="/content/drive/MyDrive/QML/validation/quantum_features_partial_windowed.csv"
):
    import numpy as np
    import pandas as pd
    import os
    import json
    import hashlib
    from tqdm import tqdm

    # Step 1: Round + deduplicate
    X_classical = df[gait_features].values
    X_classical_rounded = np.round(X_classical, 6)
    X_unique, idx_unique, idx_inverse = np.unique(
        X_classical_rounded, axis=0, return_index=True, return_inverse=True
    )

    print(f"🔍 Unique samples: {len(X_unique)} / {len(X_classical)} total")

    # Step 2: Load or initialize cache
    if os.path.exists(cache_file):
        with open(cache_file, "r") as f:
            cache = json.load(f)
    else:
        cache = {}

    # Step 3: Hashing function
    def hash_input(arr):
        return hashlib.md5(np.round(arr, 6).tobytes()).hexdigest()

    # Step 4: Safe call with cache
    def cached_quantum_call(features):
        key = hash_input(features)
        if key in cache:
            return cache[key]
        try:
            vec = quantum_feature_fn(features)
            cache[key] = vec
            with open(cache_file, "w") as f:
                json.dump(cache, f)
            return vec
        except Exception as e:
            print(f"❌ Error processing {key}: {e}")
            return [0.0] * (2 ** num_qubits)

    # Step 5: Extract features with incremental saving
    X_quantum_unique = []
    for i, x in enumerate(tqdm(X_unique)):
        X_quantum_unique.append(cached_quantum_call(x))

        # Save progress every N samples
        if (i + 1) % save_every == 0 or (i + 1) == len(X_unique):
            print(f"💾 Saving progress at sample {i + 1}")
            # Get mask of which unique rows were processed
            processed_mask = np.isin(idx_inverse, range(i + 1))

            # Reconstruct those rows
            partial_full = np.array(X_quantum_unique)[idx_inverse[processed_mask]]

            partial_df = pd.DataFrame(
                partial_full,
                columns=[f"Quantum_Feature_{j}" for j in range(2 ** num_qubits)]
            )
            partial_df["Label"] = df["Label"].values[processed_mask]

            # Save it
            partial_df.to_csv(output_csv, index=False)


    # Step 6: Reconstruct full dataset
    X_quantum_full = np.array(X_quantum_unique)[idx_inverse]

    # Step 7: Final DataFrame
    df_quantum = pd.DataFrame(
        X_quantum_full, columns=[f"Quantum_Feature_{i}" for i in range(2 ** num_qubits)]
    )
    df_quantum["Label"] = df["Label"].values

    # Final save
    df_quantum.to_csv(output_csv, index=False)
    print(f"✅ Final full CSV saved: {output_csv}")

    return df_quantum

In [ ]:
gf = df.drop(columns=["Label"])

gait_features = gf.columns.tolist()

In [ ]:
gait_features

['HipAngles_t0',
 'KneeAngles_t0',
 'AnkleAngles_t0',
 'HipAngles_t1',
 'KneeAngles_t1',
 'AnkleAngles_t1',
 'HipAngles_t2',
 'KneeAngles_t2',
 'AnkleAngles_t2',
 'HipAngles_t3',
 'KneeAngles_t3',
 'AnkleAngles_t3',
 'HipAngles_t4',
 'KneeAngles_t4',
 'AnkleAngles_t4',
 'HipAngles_t5',
 'KneeAngles_t5',
 'AnkleAngles_t5',
 'HipAngles_t6',
 'KneeAngles_t6',
 'AnkleAngles_t6',
 'HipAngles_t7',
 'KneeAngles_t7',
 'AnkleAngles_t7',
 'HipAngles_t8',
 'KneeAngles_t8',
 'AnkleAngles_t8',
 'HipAngles_t9',
 'KneeAngles_t9',
 'AnkleAngles_t9',
 'HipAngles_t10',
 'KneeAngles_t10',
 'AnkleAngles_t10',
 'HipAngles_t11',
 'KneeAngles_t11',
 'AnkleAngles_t11']

In [ ]:
df_quantum = extract_quantum_features_deduplicated(
    df=df,
    gait_features=gait_features,
    quantum_feature_fn=quantum_feature_vector,
    num_qubits=5,
    cache_file = "/content/drive/MyDrive/QML/validation/stroke_quantum_cache.json",
    save_every=10,
    output_csv="/content/drive/MyDrive/QML/validation/stroke_quantum_features_partial_windowed.csv"
)


🔍 Unique samples: 16500 / 16500 total


  0%|          | 0/16500 [00:00<?, ?it/s]

💾 Saving progress at sample 10


  0%|          | 80/16500 [00:00<01:34, 173.15it/s]

💾 Saving progress at sample 20
💾 Saving progress at sample 30
💾 Saving progress at sample 40
💾 Saving progress at sample 50
💾 Saving progress at sample 60
💾 Saving progress at sample 70
💾 Saving progress at sample 80
💾 Saving progress at sample 90
💾 Saving progress at sample 100
💾 Saving progress at sample 110
💾 Saving progress at sample 120
💾 Saving progress at sample 130
💾 Saving progress at sample 140
💾 Saving progress at sample 150


...

💾 Saving progress at sample 16490


100%|█████████▉| 16499/16500 [9:52:05<00:05,  5.72s/it]

💾 Saving progress at sample 16500


100%|██████████| 16500/16500 [9:52:12<00:00,  2.15s/it]


✅ Final full CSV saved: /content/drive/MyDrive/QML/validation/stroke_quantum_features_partial_windowed.csv


In [ ]:
df_quantum.head()

,Quantum_Feature_0,Quantum_Feature_1,Quantum_Feature_2,Quantum_Feature_3,Quantum_Feature_4,Quantum_Feature_5,Quantum_Feature_6,Quantum_Feature_7,Quantum_Feature_8,Quantum_Feature_9,...,Quantum_Feature_23,Quantum_Feature_24,Quantum_Feature_25,Quantum_Feature_26,Quantum_Feature_27,Quantum_Feature_28,Quantum_Feature_29,Quantum_Feature_30,Quantum_Feature_31,Label
0,0.023990,0.004696,0.060631,0.136102,0.139476,0.027302,0.004812,0.010803,3.351136e-03,6.559673e-04,...,9.895821e-04,0.003190,0.016297,0.092455,0.041187,0.040191,0.205322,0.015903,0.007084,0
1,0.015243,0.000057,0.001407,0.229707,0.237190,0.000888,0.000059,0.009622,2.773385e-08,1.038693e-10,...,1.328477e-10,0.000046,0.012307,0.185451,0.001136,0.001100,0.293793,0.011918,0.000073,0
2,0.003019,0.000392,0.024669,0.177140,0.184072,0.023892,0.000407,0.002924,6.071456e-04,7.880574e-05,...,8.162856e-05,0.000393,0.003029,0.177707,0.024748,0.023816,0.183485,0.002915,0.000406,0
3,0.000080,0.000091,0.041820,0.057055,0.060036,0.068404,0.000095,0.000130,1.246212e-04,1.419906e-04,...,1.138858e-04,0.000119,0.000104,0.074852,0.054864,0.052140,0.045762,0.000099,0.000073,0
4,0.000002,0.000016,0.012224,0.004137,0.004425,0.040800,0.000018,0.000006,2.558194e-05,2.358799e-04,...,1.335295e-04,0.000031,0.000003,0.007816,0.023096,0.021594,0.002342,0.000003,0.000009,0
